In [ ]:
# ============================================================
# INSTALL LIBRARIES
# ============================================================
!pip install groq pandas openpyxl scikit-learn -q

# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import json
import os

from groq import Groq

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from google.colab import files

# ============================================================
# API KEY
# ============================================================
os.environ["GROQ_API_KEY"] = "Add your API here"

# ============================================================
# UPLOAD DATASET
# ============================================================
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# ============================================================
# LOAD DATASET
# ============================================================
df = pd.read_excel(file_name, engine='openpyxl')

df.columns = df.columns.str.strip()

df = df.fillna("")

# ============================================================
# REQUIRED COLUMNS
# ============================================================
df["Ambiguous_Question"] = (
    df["Ambiguous_Question"]
    .astype(str)
    .str.strip()
)

df["Non_Ambiguous_Question"] = (
    df["Non_Ambiguous_Question"]
    .astype(str)
    .str.strip()
)

df["Label"] = (
    df["Label"]
    .astype(str)
    .str.strip()
)

# ============================================================
# FILTER LABELS
# ============================================================
df = df[df["Label"].isin(["l", "n"])]

print("Dataset Size:", len(df))

# ============================================================
# GROQ CLIENT
# ============================================================
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# ============================================================
# MAP FUNCTION
# ============================================================
def map_decision(decision):

    decision = decision.lower()

    if "not" in decision:
        return "n"

    return "l"

# ============================================================
# ACTOR LLM
# ============================================================
def actor_detect_ambiguity(query):

    prompt = f"""
You are an expert in lexical ambiguity detection.

Tasks:
1. Identify ambiguous words.
2. List meanings.
3. Decide ambiguity.
4. Give confidence.

Return STRICT JSON:

{{
  "final_decision": "Ambiguous / Not Ambiguous",
  "confidence": 0.95
}}

Sentence:
"{query}"
"""

    try:

        response = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.1
        )

        content = response.choices[0].message.content.strip()

        start = content.find("{")

        end = content.rfind("}") + 1

        result = json.loads(content[start:end])

        decision = result.get(
            "final_decision",
            "Not Ambiguous"
        )

        confidence = float(
            result.get("confidence", 0.5)
        )

        label = map_decision(decision)

        return label, confidence

    except:

        return "n", 0.0

# ============================================================
# STORAGE
# ============================================================
y_true = []

y_pred = []

confidences = []

# ============================================================
# MAIN LOOP
# ============================================================
for index, row in df.iterrows():

    query = row["Ambiguous_Question"]

    true_label = row["Label"]

    pred_label, confidence = actor_detect_ambiguity(query)

    y_true.append(true_label)

    y_pred.append(pred_label)

    confidences.append(confidence)

    print(f"Processed {index+1}/{len(df)}")

# ============================================================
# SAVE RESULTS
# ============================================================
df["Predicted_Label"] = y_pred

df["Actor_Confidence"] = confidences

# ============================================================
# METRICS
# ============================================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["l", "n"]
)

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n==============================")
print("ACTOR RESULTS")
print("==============================")

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1 Score:", f1)

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(cm)

# ============================================================
# SAVE EXCEL
# ============================================================
output_file = "actor_results.xlsx"

df.to_excel(
    output_file,
    index=False
)

print("\nSaved:", output_file)

files.download(output_file)

Saving lexical ambiguity.xlsx to lexical ambiguity (6).xlsx
Dataset Size: 139
Processed 1/139
Processed 2/139
Processed 3/139
Processed 4/139
Processed 5/139
Processed 6/139
Processed 7/139
Processed 8/139
Processed 9/139
Processed 10/139
Processed 11/139
Processed 12/139
Processed 13/139
Processed 14/139
Processed 15/139
Processed 16/139
Processed 17/139
Processed 18/139
Processed 19/139
Processed 20/139
Processed 21/139
Processed 22/139
Processed 23/139
Processed 24/139
Processed 25/139
Processed 26/139
Processed 27/139
Processed 28/139
Processed 29/139
Processed 30/139
Processed 31/139
Processed 32/139
Processed 33/139
Processed 34/139
Processed 35/139
Processed 36/139
Processed 37/139
Processed 38/139
Processed 39/139
Processed 40/139
Processed 41/139
Processed 42/139
Processed 43/139
Processed 44/139
Processed 45/139
Processed 46/139
Processed 47/139
Processed 48/139
Processed 49/139
Processed 50/139
Processed 51/139
Processed 52/139
Processed 53/139
Processed 54/139
Processed 55/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# INSTALL LIBRARIES
# ============================================================
!pip install groq pandas openpyxl sentence-transformers scikit-learn -q

# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import json
import numpy as np
import os

from groq import Groq

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from google.colab import files

# ============================================================
# API KEY
# ============================================================
os.environ["GROQ_API_KEY"] = "Add your API here"

# ============================================================
# UPLOAD ACTOR RESULTS FILE
# ============================================================
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# ============================================================
# LOAD FILE
# ============================================================
df = pd.read_excel(
    file_name,
    engine='openpyxl'
)

df.columns = df.columns.str.strip()

# ============================================================
# GROQ CLIENT
# ============================================================
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# ============================================================
# EMBEDDING MODEL
# ============================================================
embedder = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

# ============================================================
# SIMILARITY FUNCTION
# ============================================================
def semantic_similarity(text1, text2):

    emb1 = embedder.encode([text1])

    emb2 = embedder.encode([text2])

    sim = cosine_similarity(
        emb1,
        emb2
    )[0][0]

    return float(sim)

# ============================================================
# CRITIC LLM
# ============================================================
def critic_rewrite(query):

    prompt = f"""
You are a Critic LLM.

Tasks:
1. Rewrite clearly
2. Remove ambiguity
3. Preserve meaning
4. Give confidence

Return STRICT JSON:

{{
  "rewritten_query": "",
  "confidence": 0.95
}}

Query:
"{query}"
"""

    try:

        response = client.chat.completions.create(
            model="qwen/qwen3-32b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2
        )

        content = response.choices[0].message.content.strip()

        start = content.find("{")

        end = content.rfind("}") + 1

        result = json.loads(content[start:end])

        return (
            result.get("rewritten_query", query),
            float(result.get("confidence", 0.5))
        )

    except:

        return query, 0.0

# ============================================================
# STORAGE
# ============================================================
rewritten_queries = []

semantic_scores = []

rewrite_confidences = []

rewrite_correct_flags = []

# ============================================================
# MAIN LOOP
# ============================================================
for index, row in df.iterrows():

    predicted_label = row["Predicted_Label"]

    query = row["Ambiguous_Question"]

    ground_truth = row["Non_Ambiguous_Question"]

    # ========================================================
    # ONLY REWRITE AMBIGUOUS
    # ========================================================
    if predicted_label == "l":

        rewritten_query, confidence = critic_rewrite(query)

        similarity = semantic_similarity(
            rewritten_query,
            ground_truth
        )

        if similarity >= 0.85:

            correct = 1

        else:

            correct = 0

    else:

        rewritten_query = query

        confidence = 1.0

        similarity = 1.0

        correct = 1

    rewritten_queries.append(rewritten_query)

    rewrite_confidences.append(confidence)

    semantic_scores.append(similarity)

    rewrite_correct_flags.append(correct)

    print(f"Processed {index+1}/{len(df)}")

# ============================================================
# SAVE RESULTS
# ============================================================
df["Rewritten_Query"] = rewritten_queries

df["Rewrite_Confidence"] = rewrite_confidences

df["Semantic_Similarity"] = semantic_scores

df["Rewrite_Correct"] = rewrite_correct_flags

# ============================================================
# METRICS
# ============================================================
avg_similarity = np.mean(
    semantic_scores
)

rewrite_accuracy = np.mean(
    rewrite_correct_flags
)

avg_confidence = np.mean(
    rewrite_confidences
)

failure_rate = 1 - rewrite_accuracy

# ============================================================
# PRINT RESULTS
# ============================================================
print("\n==============================")
print("CRITIC RESULTS")
print("==============================")

print(
    "Average Semantic Similarity:",
    avg_similarity
)

print(
    "Rewrite Accuracy:",
    rewrite_accuracy
)

print(
    "Average Rewrite Confidence:",
    avg_confidence
)

print(
    "Rewrite Failure Rate:",
    failure_rate
)

# ============================================================
# SAVE FINAL FILE
# ============================================================
output_file = "critic_results.xlsx"

df.to_excel(
    output_file,
    index=False
)

print("\nSaved:", output_file)

files.download(output_file)

Saving llama 4 actor_results.xlsx to llama 4 actor_results (3).xlsx


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processed 1/139
Processed 2/139
Processed 3/139
Processed 4/139
Processed 5/139
Processed 6/139
Processed 7/139
Processed 8/139
Processed 9/139
Processed 10/139
Processed 11/139
Processed 12/139
Processed 13/139
Processed 14/139
Processed 15/139
Processed 16/139
Processed 17/139
Processed 18/139
Processed 19/139
Processed 20/139
Processed 21/139
Processed 22/139
Processed 23/139
Processed 24/139
Processed 25/139
Processed 26/139
Processed 27/139
Processed 28/139
Processed 29/139
Processed 30/139
Processed 31/139
Processed 32/139
Processed 33/139
Processed 34/139
Processed 35/139
Processed 36/139
Processed 37/139
Processed 38/139
Processed 39/139
Processed 40/139
Processed 41/139
Processed 42/139
Processed 43/139
Processed 44/139
Processed 45/139
Processed 46/139
Processed 47/139
Processed 48/139
Processed 49/139
Processed 50/139
Processed 51/139
Processed 52/139
Processed 53/139
Processed 54/139
Processed 55/139
Processed 56/139
Processed 57/139
Processed 58/139
Processed 59/139
Proces

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>